# ProXtal-LM — Inference & Evaluation

Interactive notebook for running inference on test data, visualising
predictions, and computing evaluation metrics.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from scripts.inference import (
    load_config_from_json,
    build_model_from_config,
    load_checkpoint_weights,
    predict_dataset,
    predict_single,
)
from scripts.evaluate_test import evaluate_test, print_metrics
from scripts.publication_graphs import apply_publication_style

apply_publication_style()

## 1. Setup

In [ ]:
CONFIG_PATH     = '../configs/large.json'
CHECKPOINT_PATH = '../checkpoints/v8_esmc_0/best_checkpoint.pt'
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

## 2. Load Model

In [ ]:
config = load_config_from_json(CONFIG_PATH)
device = torch.device(DEVICE)

model = build_model_from_config(config, device)
ckpt = load_checkpoint_weights(model, CHECKPOINT_PATH, device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')
print(f'Epoch:      {ckpt.get("epoch", "?")}')
print(f'Best loss:  {ckpt.get("best_val_loss", "?")}')

## 3. Run Inference on Test Set

In [ ]:
from proxtal_lm.data import CrystalContactsDataset, collate_pad

test_ds = CrystalContactsDataset(config.data.test_path, B=config.data.num_bins)
test_loader = DataLoader(
    test_ds,
    batch_size=config.data.batch_size,
    shuffle=False,
    num_workers=config.data.num_workers,
    collate_fn=lambda batch: collate_pad(batch, pad_multiple=config.data.pad_multiple),
)

print(f'Test samples: {len(test_ds)}')

results = predict_dataset(model, test_loader, device)
print(f'Predictions:  {len(results)}')

## 4. Visualise Predictions

In [ ]:
def plot_prediction(result, idx=0):
    """Plot predicted contact probability and distance-bin maps."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    im = ax.imshow(result['contact_prob'], cmap='hot_r', vmin=0, vmax=1)
    ax.set_title(f'Sample {idx}: Contact Probability')
    ax.set_xlabel('Residue j')
    ax.set_ylabel('Residue i')
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1]
    im = ax.imshow(result['pred_bins'], cmap='viridis')
    ax.set_title(f'Sample {idx}: Predicted Distance Bin')
    ax.set_xlabel('Residue j')
    ax.set_ylabel('Residue i')
    plt.colorbar(im, ax=ax, shrink=0.8)

    fig.tight_layout()
    plt.show()


# Show first 3 predictions
for i in range(min(3, len(results))):
    plot_prediction(results[i], idx=i)

## 5. Full Test-Set Evaluation

In [ ]:
metrics = evaluate_test(
    checkpoint_path=CHECKPOINT_PATH,
    config_path=CONFIG_PATH,
    device_str=DEVICE,
)

print_metrics(metrics)

## 6. Metrics Summary Table

In [ ]:
import pandas as pd

df_metrics = pd.DataFrame(
    [(k, v) for k, v in sorted(metrics.items())],
    columns=['Metric', 'Value'],
)
df_metrics.style.format({'Value': '{:.6f}'})

## 7. Single-Protein Inference (Example)

In [ ]:
# Load a single embedding from the test set
import h5py

with h5py.File(config.data.test_path, 'r') as f:
    keys = list(f.keys())
    sample_key = keys[0]
    emb = torch.from_numpy(f[sample_key]['embedding'][:])

    chain_id = None
    if 'chain_id' in f[sample_key]:
        chain_id = torch.from_numpy(f[sample_key]['chain_id'][:])

    sg = None
    if 'space_group' in f[sample_key].attrs:
        sg = int(f[sample_key].attrs['space_group'])

print(f'Embedding shape: {emb.shape}')

pred = predict_single(model, emb, device, chain_id=chain_id, space_group=sg)
plot_prediction(pred, idx=sample_key)